# 📌 EXEMPLO — Pipeline RAG Mínimo Viável

**Curso:** MBA RAG & CAG Aplicados a Direito e Segurança Pública  
**Aula:** 1 — Fundamentos  
**Tipo:** Exemplo demonstrativo (não é laboratório avaliado)

Este exemplo conecta tudo que foi aprendido na teoria e nos labs:
1. Carrega corpus jurídico do dataset da aula
2. Gera embeddings com BGE-M3
3. Indexa em FAISS (índice vetorial simples)
4. Executa busca semântica por uma query
5. Envia contexto + query para vLLM (Llama 3.1 8B)
6. Registra o trace no LangFuse

Este é o **Pipeline RAG mínimo** que será expandido ao longo de todas as 12 aulas.

In [2]:
import torch
import sys

print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.version.cuda}")

Python: 3.12.13
PyTorch: 2.10.0+cu128
CUDA: 12.8


In [2]:
!pip install vllm==0.17.0 --extra-index-url https://download.pytorch.org/whl/cu128

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 432.9/432.9 MB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.6/192.6 kB 21.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 130.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 133.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 114.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.9/34.9 MB 68.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7

In [1]:
# INSTALAÇÃO (execute uma vez)
!pip install sentence-transformers FlagEmbedding faiss-cpu openai langfuse pyngrok nest-asyncio

  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 110.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 479.5/479.5 kB 44.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.2/5.2 MB 141.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 79.6 MB/s eta 0:00:00
  Created wheel for warc3-wet-clueweb09: filename=warc3_wet_clueweb09-0.2.5-py3-none-any.whl size=18996 sha256=2d13e9ba73e82abc27146b9a8d29da1f5e5671708c3f57bbc86d6598b16d5e5f
  Stored in directory: /root/.cache/pip/wheels/f6/85/c2/9f0f621def52a1d5db7d29984f81e45f9fb6dfeb1a4eb6e31c


In [3]:
import json
import numpy as np
import requests
import os
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import faiss
from openai import OpenAI

print('✅ Dependências prontas!')

✅ Dependências prontas!


In [7]:
# ─── PASSO 1: Carrega corpus jurídico ────────────────────────
# Em produção: carrega de banco de dados, S3, sistema de arquivos, etc.

CORPUS_URL = 'https://raw.githubusercontent.com/tornis/ibmec-mba-rag/refs/heads/main/aula1/datasets/corpus_juridico_aula1.json'

# Corpus simplificado embutido (para demo sem internet)
corpus_local = [
    {'id': '1', 'titulo': 'Peculato doloso - servidor público',
     'conteudo': 'Servidor desviou R$2.5M do SUS. Condenado pelo art. 312 CP.'},
    {'id': '2', 'titulo': 'Habeas corpus - prisão preventiva',
     'conteudo': 'Prisão preventiva sem fundamentação idônea. Ordem concedida.'},
    {'id': '3', 'titulo': 'Furto qualificado por escalada',
     'conteudo': 'Réu entrou pela janela, subtraiu notebook e celulares.'},
    {'id': '4', 'titulo': 'Laudo balístico - homicídio',
     'conteudo': 'Projéteis 9mm da mesma arma. GSR nas mãos do suspeito.'},
    {'id': '5', 'titulo': 'Absolvição por legítima defesa',
     'conteudo': 'Réu absolvido. Vítima atacou primeiro com faca. Art. 25 CP.'},
]

corpus = []
try:
    response = requests.get(CORPUS_URL)
    response.raise_for_status() # Raise an exception for bad status codes
    corpus = json.loads(response.text)
    print(f'Corpus baixado da URL: {CORPUS_URL}')
except requests.exceptions.RequestException as e:
    print(f'Falha ao baixar corpus da URL: {e}. Usando corpus local.')
    corpus = corpus_local
except json.JSONDecodeError as e:
    print(f'Falha ao decodificar JSON: {e}. Usando corpus local.')
    corpus = corpus_local


# Combina título + conteúdo para embedding mais rico
textos = [f"{d['titulo']}. {d['conteudo']}" for d in corpus]
print(f'📚 Corpus carregado: {len(textos)} documentos')

Corpus baixado da URL: https://raw.githubusercontent.com/tornis/ibmec-mba-rag/refs/heads/main/aula1/datasets/corpus_juridico_aula1.json
📚 Corpus carregado: 10 documentos


In [8]:
# ─── PASSO 2: Carrega modelo ────────────────────────────────
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'

print('⏳ Carregando BGE-M3...')
encoder = SentenceTransformer('BAAI/bge-m3', device=DEVICE)

⏳ Carregando BGE-M3...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [9]:
# ─── PASSO 2.1: Gerar Embeddings ────────────────────────────────
print('🔢 Gerando embeddings do corpus...')
embeddings_corpus = encoder.encode(
    textos,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(f'✅ Embeddings gerados: shape={embeddings_corpus.shape}')

🔢 Gerando embeddings do corpus...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Embeddings gerados: shape=(10, 1024)


In [10]:
# ─── PASSO 2.2: Imprimir Embeddings Gerados ────────────────────────────────
print('\nExemplo do primeiro vetor de embedding (primeiras 10 dimensões):')
print(embeddings_corpus[0][:50])


Exemplo do primeiro vetor de embedding (primeiras 10 dimensões):
[-0.04875615  0.01289271 -0.02310656 -0.01482335 -0.02479001  0.00175302
 -0.01759385 -0.01035709  0.01072375 -0.00449441 -0.00804514  0.03742784
 -0.00421443 -0.01763856  0.01330962 -0.0051897   0.00472129 -0.03016805
  0.00415422 -0.00836277  0.05068494  0.01632933 -0.00181072  0.06612632
 -0.02544562  0.01129707  0.00764038 -0.02918498  0.00666552  0.01173388
 -0.04253853 -0.0462511   0.00061274 -0.01461426 -0.041449   -0.01210819
 -0.04141292 -0.01925334 -0.01746429  0.0250311   0.03248712  0.02273954
 -0.0394722  -0.07724958  0.0457705  -0.0092556  -0.00201977 -0.02212945
 -0.02034497 -0.02470318]


In [11]:
# ─── PASSO 3: Indexa no FAISS ────────────────────────────────
# FAISS = Facebook AI Similarity Search
# IndexFlatIP = produto interno (equivalente a coseno para vetores normalizados)
dimensao = embeddings_corpus.shape[1]  # 1024 para BGE-M3
indice_faiss = faiss.IndexFlatIP(dimensao)
indice_faiss.add(embeddings_corpus.astype('float32'))

print(f'📦 Índice FAISS criado!')
print(f'   Dimensão: {dimensao}')
print(f'   Documentos indexados: {indice_faiss.ntotal}')

📦 Índice FAISS criado!
   Dimensão: 1024
   Documentos indexados: 10


In [12]:
# ─── PASSO 4: Function Retrieval — Busca semântica ────────────────────

def busca_semantica(query, top_k=3):
    """Busca os top_k documentos mais relevantes para a query."""
    # Encoda a query
    query_emb = encoder.encode([query], normalize_embeddings=True)

    # Busca no índice FAISS
    scores, indices = indice_faiss.search(
        query_emb.astype('float32'),
        top_k
    )

    # Monta resultado
    resultados = []
    for score, idx in zip(scores[0], indices[0]):
        resultados.append({
            'documento': corpus[idx],
            'score': float(score),
            'texto': textos[idx]
        })
    return resultados

In [13]:
# ─── PASSO 4.1: Retrieval — Busca semântica ────────────────────
QUERY_TESTE = 'crime de servidor público que desviou dinheiro do governo'

print(f'🔍 Query: "{QUERY_TESTE}"')
print('=' * 60)

docs_recuperados = busca_semantica(QUERY_TESTE, top_k=3)

for i, doc in enumerate(docs_recuperados, 1):
    print(f'\n  [{i}] Score: {doc["score"]:.3f}')
    print(f'       Título: {doc["documento"]["titulo"]}')
    print(f'       Trecho: {doc["documento"]["conteudo"][:80]}...')

🔍 Query: "crime de servidor público que desviou dinheiro do governo"

  [1] Score: 0.643
       Título: Acórdão - Crime de Peculato Doloso - Servidor Público da Saúde
       Trecho: EMENTA: PENAL E PROCESSUAL PENAL. RECURSO ESPECIAL. PECULATO DOLOSO. SERVIDOR PÚ...

  [2] Score: 0.435
       Título: Portaria - Procedimento de Investigação - Tráfico Internacional
       Trecho: PORTARIA Nº 123/2024/GABIN/DPF. O Superintendente Regional da Polícia Federal no...

  [3] Score: 0.424
       Título: Acórdão - Estelionato - Golpe do Bilhete Premiado - Idoso
       Trecho: EMENTA: ESTELIONATO (ART. 171 CP). VÍTIMA IDOSA. CAUSA DE AUMENTO (ART. 171, §4º...


In [14]:
import subprocess
import time
import sys

# Comando configurado para rodar em background e limitar GPU
# modelos suportados para rodar no Colab
# casperhansen/deepseek-r1-distill-qwen-7b-awq
# casperhansen/llama-3-8b-instruct-awq
command = [
    "python", "-m", "vllm.entrypoints.openai.api_server",
    "--model", "casperhansen/llama-3-8b-instruct-awq",
    "--port", "8000",
    "--dtype", "half",
    "--gpu-memory-utilization", "0.70",
    "--quantization", "awq",
    "--max-model-len", "4096"
]

print("🚀 Disparando servidor vLLM em background...")
# Popen não bloqueia a execução da célula
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Pequena pausa apenas para garantir que o processo não falhou imediatamente
time.sleep(2)

if process.poll() is None:
    print("✅ Servidor rodando em segundo plano. Você pode executar as próximas células.")
else:
    print("⚠️ O processo terminou precocemente. Verifique se há erros de memória ou dependências.")

🚀 Disparando servidor vLLM em background...
✅ Servidor rodando em segundo plano. Você pode executar as próximas células.


In [20]:
import threading

log_buffer = []

def log_reader(pipe):
    with pipe:
        for line in iter(pipe.readline, ''):
            log_buffer.append(line.strip())
            # Mantém apenas as últimas 100 linhas na memória para não gastar RAM
            if len(log_buffer) > 100:
                log_buffer.pop(0)

thread = threading.Thread(target=log_reader, args=(process.stdout,), daemon=True)
thread.start()
print("🚀 Servidor vLLM disparado. Os logs aparecerão abaixo em tempo real...")
print("\n".join(log_buffer[-20:]))

🚀 Servidor vLLM disparado. Os logs aparecerão abaixo em tempo real...



In [15]:
import requests
import time

url = "http://localhost:8000/health"

for i in range(15):
    try:
        response = requests.get(url)
        if response.status_code == 200:
            print("🔥 Sucesso! O servidor vLLM está online e respondendo.")
            break
    except requests.exceptions.ConnectionError:
        print(f"⏳ Aguardando o modelo carregar... (Tentativa {i+1}/10)")
        time.sleep(5)
else:
    print("❌ O servidor não respondeu após o tempo limite.")

⏳ Aguardando o modelo carregar... (Tentativa 1/10)
⏳ Aguardando o modelo carregar... (Tentativa 2/10)
⏳ Aguardando o modelo carregar... (Tentativa 3/10)
⏳ Aguardando o modelo carregar... (Tentativa 4/10)
⏳ Aguardando o modelo carregar... (Tentativa 5/10)
⏳ Aguardando o modelo carregar... (Tentativa 6/10)
⏳ Aguardando o modelo carregar... (Tentativa 7/10)
⏳ Aguardando o modelo carregar... (Tentativa 8/10)
⏳ Aguardando o modelo carregar... (Tentativa 9/10)
⏳ Aguardando o modelo carregar... (Tentativa 10/10)
⏳ Aguardando o modelo carregar... (Tentativa 11/10)
⏳ Aguardando o modelo carregar... (Tentativa 12/10)
⏳ Aguardando o modelo carregar... (Tentativa 13/10)
🔥 Sucesso! O servidor vLLM está online e respondendo.


In [ ]:
from pyngrok import ngrok

# Substitua 'SEU_TOKEN_AQUI' pelo seu token real do Ngrok
NGROK_TOKEN = "2zEiXNBWZV96NIrGE1M7MhuPYV4_3ofdivJiPqJt4JniGcmys"
ngrok.set_auth_token(NGROK_TOKEN)

# Abre um túnel HTTP na porta 8000
public_url = ngrok.connect(8000)
print(f"Sua API do vLLM está acessível em: {public_url}")

Sua API do vLLM está acessível em: NgrokTunnel: "https://6bfe-34-87-113-198.ngrok-free.app" -> "http://localhost:8000"


In [16]:
import requests
url = "http://localhost:8000/v1/completions" # Usando a URL do Ngrok
headers = {"Content-Type": "application/json"}
data = {
    "model": "casperhansen/llama-3-8b-instruct-awq",
    "prompt": "crime de servidor público que desviou dinheiro do governo",
    "temperature": 0.7
}

response = requests.post(url, json=data)
print(response.json())

{'id': 'cmpl-b1f9fe8da7e214c1', 'object': 'text_completion', 'created': 1778005860, 'model': 'casperhansen/llama-3-8b-instruct-awq', 'choices': [{'index': 0, 'text': ' federal.\n\nEm 2017, foi preso preventivamente durante a Operação', 'logprobs': None, 'finish_reason': 'length', 'stop_reason': None, 'token_ids': None, 'prompt_logprobs': None, 'prompt_token_ids': None}], 'service_tier': None, 'system_fingerprint': None, 'usage': {'prompt_tokens': 11, 'total_tokens': 27, 'completion_tokens': 16, 'prompt_tokens_details': None}, 'kv_transfer_params': None}


In [19]:
%pip install langfuse openai --upgrade
import os
import time
from langfuse.openai import OpenAI

# Configuração de chaves e host
os.environ["LANGFUSE_PUBLIC_KEY"] = "pk-lf-829f56ba-923b-4430-8a2a-d7feb4487d08"
os.environ["LANGFUSE_SECRET_KEY"] = "sk-lf-3c4ac61a-898e-433b-a0d5-a2547dd9c66c"
os.environ["LANGFUSE_HOST"] = "https://us.cloud.langfuse.com"

In [20]:
# ─── PASSO 5: Geração com LLM + RAG ─────────────────────────
# Monta o contexto com os documentos recuperados

contexto = '\n\n'.join([
    f"[Documento {i+1}] {doc['texto']}"
    for i, doc in enumerate(docs_recuperados)
])

prompt_rag = f"""Você é um assistente jurídico especializado em direito penal brasileiro.
Responda APENAS com base nos documentos fornecidos abaixo.
Se a informação não estiver nos documentos, diga explicitamente.

DOCUMENTOS RECUPERADOS:
{contexto}

PERGUNTA: {QUERY_TESTE}

RESPOSTA FUNDAMENTADA:"""

# Chama vLLM via API OpenAI-compatível
client_llm = OpenAI(
    base_url='http://localhost:8000/v1',
    api_key='vllm'
)

print('🤖 Gerando resposta com Llama ...')
print('=' * 60)

try:
    resposta = client_llm.chat.completions.create(
        name="RAG-TESTE-VLLM-QUERY",
        model='casperhansen/llama-3-8b-instruct-awq',
        messages=[{'role': 'user', 'content': prompt_rag}],
        temperature=0.2
    )
    texto_resposta = resposta.choices[0].message.content
    print(texto_resposta)
    print(f'\n📊 Tokens: prompt={resposta.usage.prompt_tokens}, resposta={resposta.usage.completion_tokens}')
except Exception as e:
    print(f'⚠️  vLLM não disponível: {e}')
    print('   Execute os Labs 1-3 para configurar o vLLM.')
    texto_resposta = '[vLLM não configurado nesta sessão]'

🤖 Gerando resposta com Llama ...
De acordo com o Documento 1, o crime cometido pelo servidor público é o Peculato Doloso, previsto no art. 312 do Código Penal.

📊 Tokens: prompt=739, resposta=38
